# Push-T full-state PPO teacher, goal-conditioned (Jimmy): CONTINUATION of the 25M pilot

Continues the pilot (`Train_PushT_State_PPO_VarGoal`, 25M steps, test worst quadrant 56%, still rising) for another
25M steps from its `final_ckpt.pt` (weights only, fresh optimizer, same recipe). **Before running:** Add Input ->
Notebook Output Files -> the pilot notebook, so the checkpoint is under `/kaggle/input`. The step counter restarts at 0.

Trains the goal-conditioned PPO teacher for the variable-goal experiment (`CODEX_CONTEXT.md`):
random block, stick-tip and goal positions across quadrants Q1–Q4, with the goal coordinates in the observation.
Nathan's goal-conditioned BC teacher is distilled from this one. See `state_rl/README.md` for the log.

| Setting | Value |
|---|---|
| Environment | `PushTVarGoal-v1` = ManiSkill `PushT-v1` with per-episode goals (`state_rl/pusht_vargoal.py`) |
| Observation / controller | `obs_mode="state"` (31 numbers, same layout as state_bc), `pd_joint_delta_pos` (7 numbers) |
| Episode | 100 steps at 20 Hz (unchanged) |
| Training | ManiSkill's **unchanged** `ppo.py` with its PushT state recipe: 1024 envs, 100-step rollouts, 8 epochs, 32 minibatches, gamma 0.99 |
| Budget | **pilot: 25M steps** (ManiSkill's fixed-goal PushT reference); extend after looking at the curve |
| Evaluation | deterministic mean action clipped to [-1, 1]; 100 episodes per quadrant; validation seed 1001, test seed 5001 (once) |
| Freeze bar | success-once ≥ 80% in **every** quadrant on test seeds |
| Simulator | ManiSkill v3.0.1 (commit `a4a4f92`), GPU physics |

**Kaggle setup:** Settings → Accelerator → **GPU T4** (one GPU is used), Settings → **Internet on**
(needs a phone-verified account). Upload only this notebook.
For the long run use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours,
and everything in `/kaggle/working` is kept as the version's output. Interactive sessions lose files when they stop.

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, datetime, shlex, shutil, base64, hashlib

MANISKILL_COMMIT = "a4a4f9272ad64b1564035874b605ceb687b63ed8"  # v3.0.1, same as state_bc and camera PPO
ON_KAGGLE = Path("/kaggle/working").exists()
BASE = Path("/kaggle/working") if ON_KAGGLE else Path("/content")
MS_ROOT = Path("/tmp/ManiSkill")          # outside /kaggle/working so it is not saved as output
WORK_ROOT = Path("/tmp/pusht_state_ppo")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
RUN_LABEL = "vargoal_continue"

# PPO (ManiSkill's PushT state recipe; only the env id, eval settings and budget differ)
SEED = 1
TOTAL_TIMESTEPS = 25_000_000
NUM_ENVS = 1024
NUM_STEPS = 100
UPDATE_EPOCHS = 8
NUM_MINIBATCHES = 32
GAMMA = 0.99
EVAL_FREQ = 10            # iterations between eval + checkpoint (1 iteration = 102,400 steps)
NUM_EVAL_ENVS = 64
# Continue from the pilot's final checkpoint (weights only; fresh optimizer). Add the pilot notebook's
# output as an input (Add Input -> Notebook Output Files -> train-pusht-state-ppo-vargoal).
_found = sorted(Path("/kaggle/input").glob("**/vargoal_state_seed1_25M/final_ckpt.pt"))
assert _found, "Pilot checkpoint not found: add the pilot notebook's output as an input to this notebook."
RESUME_FROM = str(_found[0])
print("Resuming from:", RESUME_FROM)

# Per-quadrant evaluation and checkpoint selection
EVAL_LAST = 10            # evaluate the last N checkpoints on validation seeds
EVAL_EPISODES = 100       # per quadrant
VAL_SEED = 1001
TEST_SEED = 5001
FREEZE_BAR = 0.80

SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + RUN_LABEL
OUTPUT_ROOT = BASE / "state_rl" / SESSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOG = OUTPUT_ROOT / "notebook.log"
subprocess.run(["nvidia-smi"], check=True)
print("Platform:", "Kaggle" if ON_KAGGLE else "other", "| Notebook Python:", sys.version.split()[0])
print("Output directory:", OUTPUT_ROOT)


def run(command, **kwargs):
    """Run a command, stream its output here, append it to notebook.log, and raise on failure."""
    command = list(map(str, command))
    print("$", shlex.join(command), flush=True)
    tail = []
    with LOG.open("a") as log:
        log.write("\n$ " + shlex.join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    tail = (tail + [line.rstrip()])[-40:]
                code = process.wait()
            except BaseException:
                process.kill()
                raise
    if code:
        raise RuntimeError(f"Command failed (exit {code}). Log: {LOG}\n" + "\n".join(tail))

## 1. Install ManiSkill v3.0.1 under Python 3.11

Same pinned install as the team's Colab notebooks (torch 2.6.0 + CUDA 12.4, SAPIEN 3.0.3).

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libvulkan1", "libvulkan-dev", "vulkan-tools", "libglvnd-dev"])
if not (MS_ROOT / ".git").exists():
    run(["git", "clone", "https://github.com/mani-skill/ManiSkill.git", MS_ROOT])
run(["git", "-C", MS_ROOT, "checkout", "--detach", MANISKILL_COMMIT])
assert subprocess.check_output(["git", "-C", str(MS_ROOT), "rev-parse", "HEAD"], text=True).strip() == MANISKILL_COMMIT
PPO_PY = MS_ROOT / "examples/baselines/ppo/ppo.py"

run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.11"])
TRAIN_ENV = WORK_ROOT / "python311"
TRAIN_PY = TRAIN_ENV / "bin/python"
if not TRAIN_PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", TRAIN_ENV])
constraints = WORK_ROOT / "constraints.txt"
constraints.write_text("numpy==1.26.4\nopencv-python==4.11.0.86\nsapien==3.0.3\ntorch==2.6.0\n")
run([TRAIN_PY, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([TRAIN_PY, "-m", "pip", "install", "-q", "--constraint", constraints, "-e", MS_ROOT,
     "numpy", "sapien", "h5py", "tyro", "tensorboard", "imageio[ffmpeg]", "matplotlib"])

# Vulkan/EGL entries the hosted images can be missing; SAPIEN needs them even without video.
for filename, library, vulkan in [
    ("/usr/share/vulkan/icd.d/nvidia_icd.json", "libGLX_nvidia.so.0", True),
    ("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "libEGL_nvidia.so.0", False),
]:
    path = Path(filename)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        icd = {"library_path": library}
        if vulkan:
            icd["api_version"] = "1.2.155"
        path.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": icd}))
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["MS_SKIP_ASSET_DOWNLOAD_PROMPT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Kaggle T4 x2: use one GPU
run([TRAIN_PY, "-c", "import torch, sapien, mani_skill, numpy; print('torch', torch.__version__, 'CUDA', torch.version.cuda); "
     "print('mani_skill', mani_skill.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"])
(OUTPUT_ROOT / "pip-freeze.txt").write_text(subprocess.check_output([str(TRAIN_PY), "-m", "pip", "freeze"], text=True))

## 2. Write the state_rl code

Copies of `state_rl/` in the repo, generated by `state_rl/build_notebook.py`. Do not edit here.

In [ ]:
SRC = WORK_ROOT / "state_rl"
(SRC / "assets").mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /tmp/pusht_state_ppo/state_rl/pusht_vargoal.py
"""Variable-goal Push-T: ManiSkill PushT-v1 with a different target position per episode.

Registers ``PushTVarGoal-v1``. Import this module before ``gym.make``.

What changes relative to PushT-v1 (ManiSkill 3.0.1, commit a4a4f92):
  * Goal T position is sampled per episode from four quadrants around ``QUAD_ORIGIN``.
    Goal orientation stays fixed at PushT-v1's 5*pi/3.
  * The success check (``pseudo_render_intersection``) uses a goal transform per
    environment, recomputed at every reset. In PushT-v1 it is one fixed matrix, so a
    moved goal would silently be scored against the old target.
  * The T block starts anywhere in ``TEE_BOX`` (independent of the goal quadrant), with
    its distance to the goal limited to ``START_GOAL_DIST``.
  * The stick tip starts at a random point in ``TIP_BOX``, drawn from a precomputed
    library of arm poses (``assets/tip_start_qpos.npz``, built by make_tip_starts.py),
    at least ``TIP_TEE_MIN`` from the block.

What does NOT change: observation (31 numbers, same order; ``extra/goal_pos`` now
varies), action space and controller, 100-step episodes, the dense reward (it already
reads the goal T's live pose), and the success rule (overlap >= 0.90).

Quadrants, with dx = x - QUAD_ORIGIN[0] (positive = away from the robot base) and y:
  Q1: dx > 0, y > 0    Q2: dx < 0, y > 0    Q3: dx < 0, y < 0    Q4: dx > 0, y < 0
A band of ``AXIS_GAP`` around each axis is excluded so neighbouring quadrants do not
share near-identical goals.

Usage:
  env = gym.make("PushTVarGoal-v1", obs_mode="state", control_mode="pd_joint_delta_pos",
                 goal_quadrants=(1, 2, 3, 4))
  env.reset(seed=0)                                    # random goals in those quadrants
  env.reset(options=dict(goal_xy=torch.tensor([[-0.2, 0.1]])))  # fixed goal(s), [b, 2]
  env.unwrapped.goal_quadrant                          # [num_envs] quadrant of current goal
"""
from pathlib import Path

import numpy as np
import torch

from mani_skill.envs.tasks.tabletop.push_t import PushTEnv
from mani_skill.utils.registration import register_env
from mani_skill.utils.structs.pose import Pose

# ---- Workspace (world frame, metres). Robot base is at x = -0.615. ----------------
# Measured stick-tip reach at table height: x in about [-0.40, 0.05], y in about
# [-0.45, 0.45] (see README). Regions below keep >= ~0.1 m margin inside that.
QUAD_ORIGIN = (-0.156, 0.0)          # PushT-v1's goal x; centred in y
GOAL_HALF = (0.10, 0.20)             # goals in origin +- these
AXIS_GAP = (0.02, 0.03)              # excluded band around each quadrant axis
TEE_BOX = ((-0.30, -0.02), (-0.28, 0.28))   # T centre-of-mass start region
TIP_BOX = ((-0.36, 0.00), (-0.30, 0.30))    # stick-tip start region
START_GOAL_DIST = (0.08, 0.30)       # allowed T-start-to-goal distance
TIP_TEE_MIN = 0.15                   # stick tip starts at least this far from the T
N_CANDIDATES = 64                    # rejection-sampling candidates per reset

_QUAD_SIGNS = {1: (1.0, 1.0), 2: (-1.0, 1.0), 3: (-1.0, -1.0), 4: (1.0, -1.0)}
_DEFAULT_TIP_FILE = Path(__file__).parent / "assets" / "tip_start_qpos.npz"


def quadrant_of(goal_xy: torch.Tensor) -> torch.Tensor:
    """Quadrant (1-4) of goal positions [b, 2]."""
    dx = goal_xy[:, 0] - QUAD_ORIGIN[0]
    dy = goal_xy[:, 1] - QUAD_ORIGIN[1]
    q = torch.full_like(dx, 0, dtype=torch.long)
    q[(dx > 0) & (dy > 0)] = 1
    q[(dx < 0) & (dy > 0)] = 2
    q[(dx < 0) & (dy < 0)] = 3
    q[(dx > 0) & (dy < 0)] = 4
    return q


def sample_goals(b: int, quadrants, device=None) -> torch.Tensor:
    """Uniform goal positions [b, 2] in the given quadrants (each quadrant equally likely)."""
    quads = torch.tensor(list(quadrants), device=device)
    pick = quads[torch.randint(len(quads), (b,), device=device)]
    signs = torch.tensor([_QUAD_SIGNS[int(k)] for k in range(1, 5)], device=device)[pick - 1]
    lo = torch.tensor(AXIS_GAP, device=device)
    hi = torch.tensor(GOAL_HALF, device=device)
    mag = lo + torch.rand(b, 2, device=device) * (hi - lo)
    return torch.tensor(QUAD_ORIGIN, device=device) + signs * mag


def _uniform_box(shape, box, device):
    lo = torch.tensor([box[0][0], box[1][0]], device=device)
    hi = torch.tensor([box[0][1], box[1][1]], device=device)
    return lo + torch.rand(*shape, 2, device=device) * (hi - lo)


def _first_valid(valid: torch.Tensor) -> torch.Tensor:
    """Index of the first True per row of [b, K]; 0 if none (vanishingly rare with K=64)."""
    return valid.float().argmax(dim=1)


@register_env("PushTVarGoal-v1", max_episode_steps=100)
class PushTVarGoalEnv(PushTEnv):
    def __init__(self, *args, goal_quadrants=(1, 2, 3, 4), tip_start_file=None, **kwargs):
        assert set(goal_quadrants) <= {1, 2, 3, 4} and len(goal_quadrants) > 0
        self.goal_quadrants = tuple(int(q) for q in goal_quadrants)
        lib = np.load(tip_start_file or _DEFAULT_TIP_FILE)
        self._tip_qpos_np = lib["qpos"]
        self._tip_xy_np = lib["tip_xy"]
        super().__init__(*args, **kwargs)

    def _load_scene(self, options: dict):
        super()._load_scene(options)
        # One goal transform per environment instead of PushT-v1's single matrix.
        self.world_to_goal_trans = self.world_to_goal_trans.unsqueeze(0).repeat(self.num_envs, 1, 1)
        self.goal_quadrant = torch.zeros(self.num_envs, dtype=torch.long, device=self.device)
        self.tip_qpos_lib = torch.as_tensor(self._tip_qpos_np, device=self.device)
        self.tip_xy_lib = torch.as_tensor(self._tip_xy_np, device=self.device)

    def _goal_world_to_goal(self, goal_xy: torch.Tensor) -> torch.Tensor:
        b = goal_xy.shape[0]
        c, s = np.cos(self.goal_z_rot), np.sin(self.goal_z_rot)
        g = torch.zeros(b, 3, 3, device=goal_xy.device)
        g[:, 0, 0], g[:, 0, 1], g[:, 1, 0], g[:, 1, 1] = c, -s, s, c
        g[:, 2, 2] = 1.0
        g[:, 0:2, 2] = goal_xy
        return torch.linalg.inv(g)

    def _initialize_episode(self, env_idx: torch.Tensor, options: dict):
        # Parent sets up the table, default goal/T and robot rest pose; we override all three.
        super()._initialize_episode(env_idx, options)
        with torch.device(self.device):
            b = len(env_idx)

            # ---- goal ----
            goal_xy = (options or {}).get("goal_xy")
            if goal_xy is None:
                goal_xy = sample_goals(b, self.goal_quadrants, self.device)
            else:
                goal_xy = torch.as_tensor(goal_xy, dtype=torch.float32, device=self.device).reshape(-1, 2)
                if goal_xy.shape[0] == 1 and b > 1:
                    goal_xy = goal_xy.expand(b, 2)
                elif goal_xy.shape[0] == self.num_envs and b < self.num_envs:
                    goal_xy = goal_xy[env_idx]
                assert goal_xy.shape[0] == b, "goal_xy must be [1, 2], [b, 2] or [num_envs, 2]"
            goal_p = torch.zeros(b, 3)
            goal_p[:, :2] = goal_xy
            goal_p[:, 2] = 1e-3
            goal_q = torch.tensor([np.cos(self.goal_z_rot / 2), 0.0, 0.0, np.sin(self.goal_z_rot / 2)],
                                  dtype=torch.float32).expand(b, 4)
            self.goal_tee.set_pose(Pose.create_from_pq(p=goal_p, q=goal_q))
            self.world_to_goal_trans[env_idx] = self._goal_world_to_goal(goal_xy)
            self.goal_quadrant[env_idx] = quadrant_of(goal_xy)

            # ---- T block start: independent of goal quadrant, distance-limited ----
            cand = _uniform_box((b, N_CANDIDATES), TEE_BOX, self.device)          # [b, K, 2]
            d = (cand - goal_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid((d >= START_GOAL_DIST[0]) & (d <= START_GOAL_DIST[1]))
            tee_xy = cand[torch.arange(b), pick]
            tee_p = torch.zeros(b, 3)
            tee_p[:, :2] = tee_xy
            tee_p[:, 2] = 0.04 / 2 + 1e-3
            ang = torch.rand(b) * (2 * torch.pi)
            tee_q = torch.zeros(b, 4)
            tee_q[:, 0], tee_q[:, 3] = (ang / 2).cos(), (ang / 2).sin()
            self.tee.set_pose(Pose.create_from_pq(p=tee_p, q=tee_q))

            # ---- stick tip start: library pose away from the T ----
            n_lib = self.tip_xy_lib.shape[0]
            idx = torch.randint(n_lib, (b, N_CANDIDATES))
            dt = (self.tip_xy_lib[idx] - tee_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid(dt >= TIP_TEE_MIN)
            qpos = self.tip_qpos_lib[idx[torch.arange(b), pick]]
            qpos = qpos + torch.randn_like(qpos) * self.robot_init_qpos_noise
            self.agent.reset(qpos)


In [ ]:
%%writefile /tmp/pusht_state_ppo/state_rl/pusht_state_ppo.py
"""Goal-conditioned state PPO teacher for PushTVarGoal-v1 (Jimmy).

Training itself is ManiSkill's unchanged PPO baseline (examples/baselines/ppo/ppo.py at
commit a4a4f92), launched through run_ppo.py so the variable-goal env is registered.
This module holds everything around it:

  preflight  GPU checks of the variable-goal env (partial resets, per-env success check),
             plus observation_contract.json
  eval       per-quadrant closed-loop evaluation of PPO checkpoints, deterministic mean
             action clipped to [-1, 1]; selects a checkpoint on validation seeds and
             reports whether it clears the freeze bar in every quadrant

Metrics follow state_bc: success_once (reached the goal at some step) and success_at_end
(at the goal at step 100); episodes run the full 100 steps with no early reset.
Evaluation seeds: episodes for quadrant q use seed (base_seed + 1000 * q).
"""
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn

sys.path.insert(0, str(Path(__file__).resolve().parent))
import pusht_vargoal as pv  # noqa: E402  (registers PushTVarGoal-v1)

ENV_ID = "PushTVarGoal-v1"
CONTROL_MODE = "pd_joint_delta_pos"
OBS_FIELDS = [("agent/qpos", 7), ("agent/qvel", 7), ("extra/tcp_pose", 7),
              ("extra/goal_pos", 3), ("extra/obj_pose", 7)]  # same order as state_bc


def make_env(num_envs, sim_backend="physx_cuda", quadrants=(1, 2, 3, 4), **kw):
    import gymnasium as gym
    return gym.make(ENV_ID, num_envs=num_envs, obs_mode="state", control_mode=CONTROL_MODE,
                    sim_backend=sim_backend, goal_quadrants=quadrants, **kw)


# --------------------------------------------------------------------------------------
# Policy: deterministic mean of ManiSkill's state PPO actor
# --------------------------------------------------------------------------------------

class PPOActor(nn.Module):
    """actor_mean of examples/baselines/ppo/ppo.py (3 x 256 tanh)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, action_dim))

    @torch.no_grad()
    def act(self, raw_state):
        return self.actor_mean(raw_state).clamp(-1.0, 1.0)


def load_actor(path, state_dim=31, action_dim=7, device="cpu"):
    weights = torch.load(path, map_location=device, weights_only=False)
    actor = PPOActor(state_dim, action_dim)
    actor.load_state_dict({k: v for k, v in weights.items() if k.startswith("actor_mean.")})
    return actor.to(device).eval()


# --------------------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------------------

@torch.no_grad()
def rollout(env, actor, seed, max_steps=100):
    """One batch of episodes (one per env) for max_steps with no resets."""
    obs, _ = env.reset(seed=seed)
    n = obs.shape[0]
    dev = obs.device
    success_once = torch.zeros(n, dtype=torch.bool, device=dev)
    first = torch.full((n,), -1, dtype=torch.long, device=dev)
    success = success_once
    for t in range(max_steps):
        obs, _, _, _, info = env.step(actor.act(obs.float()))
        success = info["success"].bool()
        first = torch.where((first < 0) & success, torch.full_like(first, t + 1), first)
        success_once |= success
    steps = first[first > 0].float()
    return {"episodes": int(n), "seed": int(seed),
            "success_once": success_once.float().mean().item(),
            "success_at_end": success.float().mean().item(),
            "mean_steps_to_first_success": steps.mean().item() if len(steps) else None}


def eval_per_quadrant(env, actor, base_seed, quadrants=(1, 2, 3, 4), max_steps=100):
    u = env.unwrapped
    out = {}
    for q in quadrants:
        u.goal_quadrants = (q,)
        r = rollout(env, actor, base_seed + 1000 * q, max_steps)
        assert bool((u.goal_quadrant == q).all()), "goals left the requested quadrant"
        out[f"Q{q}"] = r
    u.goal_quadrants = (1, 2, 3, 4)
    so = [out[f"Q{q}"]["success_once"] for q in quadrants]
    se = [out[f"Q{q}"]["success_at_end"] for q in quadrants]
    out["worst_success_once"] = min(so)
    out["mean_success_once"] = float(np.mean(so))
    out["mean_success_at_end"] = float(np.mean(se))
    return out


def checkpoint_iteration(p: Path):
    s = p.stem.split("_")[-1]
    return int(s) if s.isdigit() else 10**9  # final_ckpt sorts last


def cmd_eval(args):
    device = torch.device(args.device)
    ckpts = sorted(Path(args.run_dir).glob("*.pt"), key=checkpoint_iteration) if args.run_dir else []
    ckpts = ckpts[-args.last:] if args.last else ckpts
    ckpts += [Path(c) for c in args.ckpt]
    assert ckpts, "no checkpoints found"
    env = make_env(args.episodes, sim_backend=args.sim_backend)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for c in ckpts:
        actor = load_actor(c, device=device)
        r = eval_per_quadrant(env, actor, args.val_seed)
        rows.append({"checkpoint": str(c), **r})
        print(f"{c.name:>18}  worst {r['worst_success_once']:.0%}  mean {r['mean_success_once']:.0%}  "
              + "  ".join(f"Q{q} {r[f'Q{q}']['success_once']:.0%}/{r[f'Q{q}']['success_at_end']:.0%}"
                          for q in (1, 2, 3, 4)), flush=True)
    (out / "val_results.json").write_text(json.dumps(rows, indent=2))
    # Selection on validation only: best worst-quadrant success_once, then mean, then later checkpoint.
    best = max(rows, key=lambda r: (r["worst_success_once"], r["mean_success_once"],
                                    checkpoint_iteration(Path(r["checkpoint"]))))
    print("\nSelected on validation:", best["checkpoint"])
    result = {"selected": best["checkpoint"], "val": best, "freeze_bar": args.bar,
              "val_seed": args.val_seed, "episodes_per_quadrant": args.episodes}
    if args.test_seed is not None:
        test = eval_per_quadrant(env, load_actor(best["checkpoint"], device=device), args.test_seed)
        result.update(test=test, test_seed=args.test_seed,
                      passes_bar=all(test[f"Q{q}"]["success_once"] >= args.bar for q in (1, 2, 3, 4)))
        print("Test (used once):  " + "  ".join(
            f"Q{q} {test[f'Q{q}']['success_once']:.0%}/{test[f'Q{q}']['success_at_end']:.0%}" for q in (1, 2, 3, 4)))
        print(f"Freeze bar {args.bar:.0%} success_once in every quadrant:",
              "PASS" if result["passes_bar"] else "FAIL")
    (out / "selection.json").write_text(json.dumps(result, indent=2))
    env.close()


# --------------------------------------------------------------------------------------
# Preflight
# --------------------------------------------------------------------------------------

def _place_tee(u, xy, ang):
    from mani_skill.utils.structs.pose import Pose
    b = xy.shape[0]
    p = torch.zeros(b, 3, device=xy.device)
    p[:, :2], p[:, 2] = xy, 0.021
    q = torch.zeros(b, 4, device=xy.device)
    q[:, 0], q[:, 3] = np.cos(ang / 2), np.sin(ang / 2)
    u.tee.set_pose(Pose.create_from_pq(p=p, q=q))


def preflight(num_envs=8, sim_backend="physx_cuda", out=None):
    env = make_env(num_envs, sim_backend=sim_backend)
    u = env.unwrapped
    obs, _ = env.reset(seed=0)
    from mani_skill.utils import gym_utils
    checks = {}

    # 1. observation layout and spaces match state_bc's contract
    raw = u._get_obs_agent()
    extra = u._get_obs_extra(u.get_info())
    parts = [raw["qpos"], raw["qvel"], extra["tcp_pose"], extra["goal_pos"], extra["obj_pose"]]
    checks["obs_layout"] = bool(torch.allclose(torch.cat(parts, -1).float(), obs.float(), atol=1e-5))
    checks["dims"] = tuple(obs.shape[1:]) == (31,) and tuple(u.single_action_space.shape) == (7,)
    checks["max_episode_steps_100"] = gym_utils.find_max_episode_steps_value(env) == 100

    # 2. every env scores success against its own goal
    goals = u.goal_tee.pose.p[:, :2].clone()
    _place_tee(u, goals, u.goal_z_rot)
    checks["success_at_own_goal"] = bool(u.evaluate()["success"].all())
    shifted = goals.roll(1, 0)
    far = (shifted - goals).norm(dim=1) > 0.04
    _place_tee(u, shifted, u.goal_z_rot)
    checks["no_success_at_other_goal"] = bool((~u.evaluate()["success"][far]).all()) if far.any() else True

    # 3. partial reset changes only the reset envs (what PPO's auto-reset does)
    if num_envs >= 2:
        env.reset(seed=1)
        before_goal = u.goal_tee.pose.p[:, :2].clone()
        before_trans = u.world_to_goal_trans.clone()
        idx = torch.arange(0, num_envs, 2, device=u.device)
        keep = torch.arange(1, num_envs, 2, device=u.device)
        env.reset(options=dict(env_idx=idx))
        after_goal = u.goal_tee.pose.p[:, :2]
        checks["partial_reset_keeps_others"] = bool(
            torch.allclose(after_goal[keep], before_goal[keep]) and
            torch.allclose(u.world_to_goal_trans[keep], before_trans[keep]))
        checks["partial_reset_changes_reset_envs"] = bool((after_goal[idx] - before_goal[idx]).norm(dim=1).gt(1e-4).all())
        _place_tee(u, after_goal.clone(), u.goal_z_rot)
        checks["success_after_partial_reset"] = bool(u.evaluate()["success"].all())
        checks["quadrant_labels_after_partial_reset"] = bool((u.goal_quadrant == pv.quadrant_of(after_goal)).all())

    # 4. quadrant restriction
    u.goal_quadrants = (4,)
    env.reset(seed=2)
    checks["quadrant_restriction"] = bool((u.goal_quadrant == 4).all())
    u.goal_quadrants = (1, 2, 3, 4)

    contract = {
        "env_id": ENV_ID, "base_env": "PushT-v1", "obs_mode": "state", "control_mode": CONTROL_MODE,
        "state_dim": 31, "action_dim": 7, "max_episode_steps": 100,
        "control_freq": u.control_freq, "sim_freq": u.sim_freq, "sim_backend": sim_backend,
        "state_fields": [{"name": n, "start": s, "stop": s + k}
                         for (n, k), s in zip(OBS_FIELDS, np.cumsum([0] + [k for _, k in OBS_FIELDS[:-1]]).tolist())],
        "action_low": u.single_action_space.low.tolist(), "action_high": u.single_action_space.high.tolist(),
        "goal_fields_to_hide_from_transformer": ["extra/goal_pos (indices 21:24)"],
        "workspace": {"quad_origin": pv.QUAD_ORIGIN, "goal_half": pv.GOAL_HALF, "axis_gap": pv.AXIS_GAP,
                      "tee_box": pv.TEE_BOX, "tip_box": pv.TIP_BOX, "start_goal_dist": pv.START_GOAL_DIST,
                      "tip_tee_min": pv.TIP_TEE_MIN, "goal_z_rot": float(u.goal_z_rot)},
        "preflight": checks,
    }
    if out:
        Path(out).mkdir(parents=True, exist_ok=True)
        (Path(out) / "observation_contract.json").write_text(json.dumps(contract, indent=2))
    env.close()
    for k, v in checks.items():
        print(f"{'PASS' if v else 'FAIL'}  {k}")
    if not all(checks.values()):
        raise SystemExit("Preflight failed")
    return contract


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("preflight")
    a.add_argument("--out", required=True)
    a.add_argument("--num-envs", type=int, default=8)
    a.add_argument("--sim-backend", default="physx_cuda")
    e = sub.add_parser("eval")
    e.add_argument("--run-dir", help="ppo.py run folder with ckpt_*.pt / final_ckpt.pt")
    e.add_argument("--ckpt", nargs="*", default=[])
    e.add_argument("--last", type=int, default=0, help="only the last N checkpoints of --run-dir")
    e.add_argument("--episodes", type=int, default=100, help="per quadrant")
    e.add_argument("--val-seed", type=int, default=1001)
    e.add_argument("--test-seed", type=int, default=None)
    e.add_argument("--bar", type=float, default=0.80)
    e.add_argument("--out", required=True)
    e.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    e.add_argument("--sim-backend", default="physx_cuda")
    args = p.parse_args(argv)
    if args.cmd == "preflight":
        preflight(args.num_envs, args.sim_backend, args.out)
    else:
        cmd_eval(args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/pusht_state_ppo/state_rl/run_ppo.py
"""Run ManiSkill's unchanged PPO baseline with PushTVarGoal-v1 registered.

Usage: python state_rl/run_ppo.py /path/to/ManiSkill/examples/baselines/ppo/ppo.py --env_id=PushTVarGoal-v1 [ppo.py args]
"""
import runpy
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import pusht_vargoal  # noqa: E402,F401  registers PushTVarGoal-v1

ppo_path = sys.argv[1]
sys.argv = [ppo_path] + sys.argv[2:]
runpy.run_path(ppo_path, run_name="__main__")


In [ ]:
# Tip-start pose library (state_rl/assets/tip_start_qpos.npz, built by make_tip_starts.py)
LIB_B64 = "..."
lib_bytes = base64.b64decode(LIB_B64)
assert hashlib.sha256(lib_bytes).hexdigest() == "1c3100a73d29aa9c811cf61a02a6c70f4c08ecb4f31ad46023eaba19b1c847d5"
(SRC / "assets" / "tip_start_qpos.npz").write_bytes(lib_bytes)
CODE_COPY = OUTPUT_ROOT / "code"
shutil.copytree(SRC, CODE_COPY, dirs_exist_ok=True)
TOOL = SRC / "pusht_state_ppo.py"
RUN_PPO = SRC / "run_ppo.py"
run([TRAIN_PY, TOOL, "--help"])

## 3. Preflight on the GPU simulator

Checks the observation layout, that every environment scores success against **its own** goal,
that partial resets (PPO's auto-reset) only change the reset environments, and the quadrant restriction.
Writes `observation_contract.json`. Stop if anything fails.

In [ ]:
run([TRAIN_PY, TOOL, "preflight", "--out", OUTPUT_ROOT, "--num-envs", 16])
contract = json.loads((OUTPUT_ROOT / "observation_contract.json").read_text())
assert all(contract["preflight"].values()) and contract["max_episode_steps"] == 100

## 4. Smoke test: tiny PPO run + evaluation

A few iterations to check the full pipeline before the long run.

In [ ]:
SMOKE = WORK_ROOT / "smoke"
SMOKE.mkdir(exist_ok=True)
run([TRAIN_PY, RUN_PPO, PPO_PY, "--env_id=PushTVarGoal-v1", "--seed=0", "--num_envs=64", "--num-steps=100",
     "--total_timesteps=12800", "--num_eval_envs=8", "--num_eval_steps=100", "--eval_freq=1",
     "--no-capture-video", "--exp-name=smoke"], cwd=SMOKE)
assert list((SMOKE / "runs/smoke").glob("*.pt")), "smoke run saved no checkpoint"
run([TRAIN_PY, TOOL, "eval", "--run-dir", SMOKE / "runs/smoke", "--last", 1, "--episodes", 8, "--out", SMOKE / "eval"])

## 5. Train

ManiSkill's `ppo.py`, unchanged. Watch `eval_success_once_mean` (all quadrants mixed) and `SPS`.
Checkpoints and TensorBoard logs go to `runs/<run name>/` in the output folder.

In [ ]:
RUN_NAME = f"vargoal_state_seed{SEED}_cont{TOTAL_TIMESTEPS // 1_000_000}M"
args = [TRAIN_PY, RUN_PPO, PPO_PY, "--env_id=PushTVarGoal-v1", f"--seed={SEED}",
        f"--num_envs={NUM_ENVS}", f"--num-steps={NUM_STEPS}", f"--update_epochs={UPDATE_EPOCHS}",
        f"--num_minibatches={NUM_MINIBATCHES}", f"--gamma={GAMMA}", f"--total_timesteps={TOTAL_TIMESTEPS}",
        f"--num_eval_envs={NUM_EVAL_ENVS}", "--num_eval_steps=100", f"--eval_freq={EVAL_FREQ}",
        "--no-capture-video", f"--exp-name={RUN_NAME}"]
if RESUME_FROM:
    args.append(f"--checkpoint={RESUME_FROM}")
(OUTPUT_ROOT / "train_command.txt").write_text(shlex.join(map(str, args)))
run(args, cwd=OUTPUT_ROOT)
RUN_DIR = OUTPUT_ROOT / "runs" / RUN_NAME

## 6. Per-quadrant evaluation, selection and freeze check

Evaluates the last checkpoints on validation seeds (100 episodes per quadrant), selects the one with the best
**worst-quadrant** success-once, then tests it once on fresh seeds against the 80% bar.

In [ ]:
EVAL_DIR = OUTPUT_ROOT / "eval"
run([TRAIN_PY, TOOL, "eval", "--run-dir", RUN_DIR, "--last", EVAL_LAST, "--episodes", EVAL_EPISODES,
     "--val-seed", VAL_SEED, "--test-seed", TEST_SEED, "--bar", FREEZE_BAR, "--out", EVAL_DIR])
sel = json.loads((EVAL_DIR / "selection.json").read_text())
shutil.copy2(sel["selected"], OUTPUT_ROOT / "teacher.pt")
print("\nTeacher candidate:", OUTPUT_ROOT / "teacher.pt", "<- copy of", Path(sel["selected"]).name)
print("Passes freeze bar:", sel["passes_bar"])

## 7. Save the results

On Kaggle, outputs persist only in a committed version (Save Version → Save & Run All).
In an interactive session, download the folder below before the session ends.
Log the results in `state_rl/README.md`.

In [ ]:
shutil.make_archive(str(OUTPUT_ROOT), "zip", OUTPUT_ROOT)
print("Folder:", OUTPUT_ROOT)
print("Zip:   ", str(OUTPUT_ROOT) + ".zip")